# Recording chains with BEN

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`gerrytools.ben` records GerryChain runs as BENDL files: compact, self-describing binary bundles
from the [binary-ensemble](https://pypi.org/project/binary-ensemble/) (BEN) family of formats. A
BENDL file carries the dual graph, optional metadata, and every step's district assignment, so a
recorded run can be re-read, subsampled, and shared without re-running the chain or shipping the
original scripts.

`RecordedChain` is a drop-in `gerrychain.MarkovChain` subclass. Iterating it yields the live
partitions exactly as a plain chain would, while each step's assignment vector streams to disk.
The file is written to a temporary location and atomically published only when the run finishes
cleanly, so a crashed run never leaves a half-written file in the destination's place.

In [ ]:
import tempfile
from pathlib import Path

import networkx as nx
from gerrychain import Partition
from gerrychain.constraints import contiguous
from gerrychain.proposals import build_recom_proposal_fn
from gerrychain.updaters import Tally

from gerrytools.ben import RecordedChain, RecordedRun

output_dir = Path(tempfile.mkdtemp())

## A small example chain

The example graph is a 6-by-6 grid with one person per node, striped into four starting districts
of nine nodes each. Any NetworkX graph or NetworkX-backed GerryChain `Graph` works the same way;
for a real ensemble this is where a state dual graph (for example from
`dualgraphs20`, see the [data overview](data/overview.ipynb)) comes in.

In [ ]:
grid = nx.grid_2d_graph(6, 6)
grid = nx.convert_node_labels_to_integers(grid, ordering="sorted")
for node in grid:
    grid.nodes[node]["population"] = 1
    grid.nodes[node]["district"] = node // 9

chain = RecordedChain(
    grid,
    output_path=output_dir / "grid.bendl",
    total_steps=50,
    rng=0,
)

Construction takes the graph and the recording settings; the chain pieces come after. That order
matters: `RecordedChain` canonicalizes and reorders the graph for compact encoding, and every
partition in the run must be built on exactly that prepared graph, exposed as `chain.graph`.
Building the initial partition from the original `grid` instead would fail the run's built-in
verification.

In [ ]:
chain.initial_partition = Partition(
    chain.graph, "district", updaters={"population": Tally("population")}
)
chain.proposal_fn = build_recom_proposal_fn("population", pop_target=9, epsilon=0.15)
chain.add_constraint(contiguous)

Iterating the chain runs it and records it. Anything you would do with a plain `MarkovChain`
works: score each partition as it is produced, collect statistics, or exhaust the iterator.

In [ ]:
cut_edge_counts = [len(partition["cut_edges"]) for partition in chain]

print("steps recorded:", len(cut_edge_counts))
print("recording size:", chain.output_path.stat().st_size, "bytes")

Fifty 36-node plans, the graph, and the format framing fit in under two kilobytes: consecutive
plans differ by small deltas, and the encoding exploits that.

## Reading a recording

A clean run exposes its reader as `chain.recording`, a `RecordedRun` bound to the published
file. `assignment_at(i)` fetches the zero-based step `i` as an assignment vector in
`chain.graph` node order; sample 0 is the initial partition. Iterating over the recording
yields every assignment in order. Use `len(recording)` or `recording.count_samples()` for
the total sample count, and call `recording.verify()` when file integrity matters.

In [ ]:
recording = chain.recording
recording.verify()
print("samples:", recording.count_samples())
print("step 10 assignment:", recording.assignment_at(10)[:12], "...")

`partition_at(i)` reconstructs a full `Partition` (of the same class and with the same
updaters as the recorded run) on the embedded graph:

In [ ]:
step_10 = recording.partition_at(10)
print(type(step_10).__name__, "with districts", sorted(step_10.parts))
print("district populations:", dict(step_10["population"]))

## Subsampling

Three iterators on the recording thin it without loading it whole, yielding assignment vectors:

- `subsample_every(step, offset=0)` for every `step`-th plan,
- `subsample_indices(indices)` for an explicit list, and
- `subsample_range(start, end)` for a half-open range.

When reconstructed plans are needed, `partitions(...)` wraps any of these (or any other
iterable of assignment vectors) and lazily yields `Partition` objects.

In [ ]:
thinned = list(recording.subsample_every(10))
print("every 10th plan:", len(thinned), "samples")

window = list(recording.partitions(recording.subsample_range(20, 25)))
print("plans 20-24 reconstructed:", [sorted(plan.parts) == [0, 1, 2, 3] for plan in window])

## Reruns and overwrite protection

A `RecordedChain` refuses to clobber its own output: iterating a second time raises, as does
pointing a fresh chain at an existing file. `allow_overwrite()` returns one authorized run
iterator that replaces the destination atomically, and a fresh `chain.recording` reader
replaces the old one when the rerun publishes.

In [ ]:
try:
    list(chain)
except RuntimeError as error:
    print("second run refused:", error)

rerun_steps = len(list(chain.allow_overwrite()))
print("authorized rerun recorded", rerun_steps, "steps")

If a run dies partway, the partial recording is preserved next to the destination (the exception
notes name the path) and the destination itself is untouched.

## Metadata, graph order, and variants

Three constructor knobs shape the file:

- `metadata=` embeds a JSON-serializable dict or list in the bundle, the natural home for the
  proposal name, epsilon, seed, and data vintages a reader needs to interpret the run.
- `graph_order=` controls node reordering before encoding: `"mlc"` (the default), `"rcm"`, a
  node-attribute `"key"` (with `graph_order_key=`), or `None` to keep the input order. Reordering
  helps compression; the permutation back to the source order is stored in the file.
- `variant=` picks the assignment-stream encoding: `"twodelta"` (the default), `"mkv_chain"`, or
  `"standard"`. See the binary-ensemble package for format details.

In [ ]:
documented = RecordedChain(
    grid,
    output_path=output_dir / "documented.bendl",
    total_steps=5,
    rng=0,
    metadata={"proposal": "recom", "epsilon": 0.15, "seed": 0},
)
documented.initial_partition = Partition(documented.graph, "district")
documented.proposal_fn = build_recom_proposal_fn("population", pop_target=9, epsilon=0.15)
documented.add_constraint(contiguous)
list(documented)

print("metadata read back:", documented.recording.metadata)

## Recorded run files stand alone

The file can be reopened as a `RecordedRun` without the original chain. Supply any updaters
that reconstructed partitions should carry; the standard `Partition` class is used by default.
For assignment vectors alone, no updaters are needed.

Nothing about lower-level reading requires gerrytools: the file embeds the graph, so
`binary_ensemble.BendlDecoder` can open it anywhere. A `RecordedRun` also exposes its decoder
as `recording.decoder` for format details and raw asset access. The
[using-BENDL guide](https://binary-ensemble.readthedocs.io/en/latest/user/using_bendl/) and
[API reference](https://binary-ensemble.readthedocs.io/en/latest/api/) in the binary-ensemble
documentation cover the decoder's full surface, the other tools that consume these files, and the
format itself.

In [ ]:
reopened = RecordedRun.from_bendl(
    output_dir / "grid.bendl",
    updaters={"population": Tally("population")},
)
print("reopened step 10:", dict(reopened.partition_at(10)["population"]))

In [ ]:
from binary_ensemble import BendlDecoder

decoder = BendlDecoder(output_dir / "grid.bendl")
embedded_graph = decoder.read_graph()
print("embedded graph:", embedded_graph)
print("node 0 attributes:", dict(embedded_graph.nodes[0]))
print("complete recording:", decoder.is_complete())

## Related

- The [MGRP runners](mgrp.md) generate large ensembles in Docker containers; `RecordedChain`
  records chains you drive from Python.
- The [plotting guides](plotting/index.md) turn recorded ensembles into distributions and maps.
- [BEN API](../api/ben.rst)